# XGBoost Training with WOE Feature Engineering

Predict `error` vs `no_error` using enriched model + question features from `data/realmistake_full_enriched.csv`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "realmistake_full_enriched.csv"
MODEL_DIR = PROJECT_ROOT / "models"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

## Load data

In [2]:
df = pd.read_csv(DATA_PATH)
df["target"] = (df["error"] == "error").astype(int)

print(f"Rows: {len(df)}")
print(df["target"].value_counts())
df.head(3)

Rows: 900
target
1    649
0    251
Name: count, dtype: int64


,question,llm_model,error,model_name,context_window_tokens,max_output_tokens,vocab_size,positional_encoding_type,attention_type,tokenizer_type,...,question_length_words,question_length_chars,question_complexity_score,has_few_shot_examples,prompt_contains_system_instructions,question_category,is_ambiguous,contains_negation,context_token_count,target
0,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,error,llama-2-70b-chat-hf,4096,2048,32000,RoPE,GQA,sentencepiece_BPE,...,274,1758,10.51,False,False,Reasoning,False,True,342,1
1,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,error,llama-2-70b-chat-hf,4096,2048,32000,RoPE,GQA,sentencepiece_BPE,...,264,1690,10.27,False,False,Reasoning,False,True,330,1
2,Generate a math word problem that satisfies th...,meta-llama/Llama-2-70b-chat-hf,no_error,llama-2-70b-chat-hf,4096,2048,32000,RoPE,GQA,sentencepiece_BPE,...,103,652,10.50,False,False,Reasoning,False,True,128,0


## Train / validation / test split

In [3]:
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=train_val_df["target"],
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df["target"].value_counts(normalize=True).round(3))

Train: 630 | Val: 135 | Test: 135
target
1    0.722
0    0.278
Name: proportion, dtype: float64


## Feature engineering and WOE encoding

In [4]:
CATEGORICAL_COLUMNS = [
    "model_name",
    "positional_encoding_type",
    "attention_type",
    "tokenizer_type",
    "question_category",
]

BOOLEAN_COLUMNS = [
    "is_open_source",
    "multilingual_support",
    "has_few_shot_examples",
    "prompt_contains_system_instructions",
    "is_ambiguous",
    "contains_negation",
]

NUMERIC_COLUMNS = [
    "context_window_tokens",
    "max_output_tokens",
    "vocab_size",
    "knowledge_cutoff_year",
    "temperature",
    "top_p",
    "top_k",
    "repetition_penalty",
    "frequency_penalty",
    "presence_penalty",
    "max_tokens_requested",
    "stop_sequences_count",
    "galileo_qa_no_rag",
    "galileo_qa_with_rag",
    "galileo_longform",
    "crag_hallucination_rate",
    "crag_accuracy",
    "question_length_words",
    "question_length_chars",
    "question_complexity_score",
    "context_token_count",
]


def compute_woe_maps(frame, columns, target="target"):
    maps = {}
    total_events = frame[target].sum()
    total_non_events = len(frame) - total_events
    for column in columns:
        grouped = frame.groupby(column, dropna=False)[target].agg(["sum", "count"])
        grouped["non_events"] = grouped["count"] - grouped["sum"]
        woe_map = {}
        for value, row in grouped.iterrows():
            event_rate = (row["sum"] + 0.5) / (total_events + 1.0)
            non_event_rate = (row["non_events"] + 0.5) / (total_non_events + 1.0)
            woe_map[value] = float(np.log(event_rate / non_event_rate))
        maps[column] = woe_map
    return maps


def apply_woe(frame, columns, maps):
    transformed = frame.copy()
    for column in columns:
        transformed[f"{column}_woe"] = transformed[column].map(maps[column]).fillna(0.0)
    return transformed


def build_feature_matrix(frame):
    numeric = frame[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    boolean = frame[BOOLEAN_COLUMNS].astype(int)
    woe_cols = [f"{col}_woe" for col in CATEGORICAL_COLUMNS]
    return pd.concat([numeric, boolean, frame[woe_cols]], axis=1)


woe_maps = compute_woe_maps(train_df, CATEGORICAL_COLUMNS)
train_df = apply_woe(train_df, CATEGORICAL_COLUMNS, woe_maps)
val_df = apply_woe(val_df, CATEGORICAL_COLUMNS, woe_maps)
test_df = apply_woe(test_df, CATEGORICAL_COLUMNS, woe_maps)

feature_names = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + [f"{col}_woe" for col in CATEGORICAL_COLUMNS]

x_train = build_feature_matrix(train_df)
x_val = build_feature_matrix(val_df)
x_test = build_feature_matrix(test_df)

medians = x_train.median(numeric_only=True)
x_train = x_train.fillna(medians)
x_val = x_val.fillna(medians)
x_test = x_test.fillna(medians)

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

print(f"Feature count: {len(feature_names)}")
x_train.head()

Feature count: 32


,context_window_tokens,max_output_tokens,vocab_size,knowledge_cutoff_year,temperature,top_p,top_k,repetition_penalty,frequency_penalty,presence_penalty,...,multilingual_support,has_few_shot_examples,prompt_contains_system_instructions,is_ambiguous,contains_negation,model_name_woe,positional_encoding_type_woe,attention_type_woe,tokenizer_type_woe,question_category_woe
159,4096,2048,32000,2022,0.6,0.9,50.0,1.2,0.0,0.0,...,0,0,0,0,1,0.527776,0.527776,0.527776,0.527776,-0.041745
29,4096,2048,32000,2022,0.6,0.9,50.0,1.2,0.0,0.0,...,0,0,0,0,1,0.527776,0.527776,0.527776,0.527776,-0.041745
138,4096,2048,32000,2022,0.6,0.9,50.0,1.2,0.0,0.0,...,0,0,0,0,1,0.527776,0.527776,0.527776,0.527776,-0.041745
426,4096,2048,32000,2022,0.6,0.9,50.0,1.2,0.0,0.0,...,0,0,0,0,1,0.527776,0.527776,0.527776,0.527776,0.110885
648,4096,2048,32000,2022,0.6,0.9,50.0,1.2,0.0,0.0,...,0,0,0,0,1,0.527776,0.527776,0.527776,0.527776,-0.041745


In [5]:
pd.DataFrame(
    {
        "feature": [f"{col}_woe" for col in CATEGORICAL_COLUMNS],
        "woe_values": [woe_maps[col] for col in CATEGORICAL_COLUMNS],
    }
)

,feature,woe_values
0,model_name_woe,"{'gpt-4-0613': -0.4825559976927863, 'llama-2-7..."
1,positional_encoding_type_woe,"{'RoPE': 0.5277758897309953, 'learned_absolute..."
2,attention_type_woe,"{'GQA': 0.5277758897309953, 'MHA': -0.48255599..."
3,tokenizer_type_woe,"{'cl100k_BPE': -0.4825559976927863, 'sentencep..."
4,question_category_woe,"{'Coding': -0.013739175883304158, 'Creative': ..."


## Train XGBoost

In [6]:
scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    verbose=False,
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

## Evaluation helper

In [7]:
def evaluate_split(name, x, y, model):
    proba = model.predict_proba(x)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "split": name,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "confusion_matrix": confusion_matrix(y, preds),
        "report": classification_report(y, preds, zero_division=0),
    }


scores = pd.DataFrame(
    [
        evaluate_split("train", x_train, y_train, model),
        evaluate_split("val", x_val, y_val, model),
        evaluate_split("test", x_test, y_test, model),
    ]
)

scores[["split", "accuracy", "precision", "recall", "f1", "roc_auc"]]

,split,accuracy,precision,recall,f1,roc_auc
0,train,0.900000,0.973430,0.885714,0.927503,0.972232
1,val,0.696296,0.785714,0.793814,0.789744,0.758003
2,test,0.748148,0.853933,0.783505,0.817204,0.740369


In [8]:
for _, row in scores.iterrows():
    print(f"{row['split'].upper()} confusion matrix:\n{row['confusion_matrix']}\n")
    print(row["report"])
    print("-" * 60)

TRAIN confusion matrix:
[[164  11]
 [ 52 403]]

              precision    recall  f1-score   support

           0       0.76      0.94      0.84       175
           1       0.97      0.89      0.93       455

    accuracy                           0.90       630
   macro avg       0.87      0.91      0.88       630
weighted avg       0.91      0.90      0.90       630

------------------------------------------------------------
VAL confusion matrix:
[[17 21]
 [20 77]]

              precision    recall  f1-score   support

           0       0.46      0.45      0.45        38
           1       0.79      0.79      0.79        97

    accuracy                           0.70       135
   macro avg       0.62      0.62      0.62       135
weighted avg       0.69      0.70      0.70       135

------------------------------------------------------------
TEST confusion matrix:
[[25 13]
 [21 76]]

              precision    recall  f1-score   support

           0       0.54      0.66   

## Inference on test set

In [ ]:
test_proba = model.predict_proba(x_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

inference_df = test_df[["llm_model", "error", "question_category"]].copy()
inference_df["predicted_error_probability"] = test_proba
inference_df["predicted_label"] = np.where(test_pred == 1, "error", "no_error")
inference_df["correct"] = inference_df["error"] == inference_df["predicted_label"]

print(f"Test accuracy from inference view: {inference_df['correct'].mean():.4f}")
inference_df.head(10)

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
importance.head(15)

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_model(MODEL_DIR / "xgboost_woe.json")

artifact = {
    "feature_names": feature_names,
    "woe_maps": {k: {str(key): val for key, val in v.items()} for k, v in woe_maps.items()},
    "numeric_medians": medians.to_dict(),
}
with (MODEL_DIR / "preprocessing.json").open("w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved model to {MODEL_DIR / 'xgboost_woe.json'}")